# Submission 2: Reflection and the Lightweight Challenge
### ME 323 Module 1

<img src="https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/me323/Module1_drafts/figures/I_beam_dimensions.jpg" alt="I-beam dimensions" width="220">

Your beam has been printed and tested. Three jobs here:

1. Recall the module's ideas from memory.
2. Reflect on your measured result against your recorded prediction.
3. Design the lightest beam that confidently clears 700 N.

## 0. Recall

Write before computing. Corrections earn credit; unsupported bluffing does not.

1. Name the three modeled capacity branches and explain how the dominant-mode
   proxy is assigned. Which region of the (b, H_web) box does each own?
2. Pre-lab 1 calibrated σ_y, k, and c_s. For each: was it a correction or a
   confession? (One sentence each.)
3. Distinguish epistemic, aleatory, and total predictive uncertainty. Which
   sigma drives explore-vs-exploit, and which belongs in a future-beam bound?
4. The equation query returned below its prediction; the GP query returned
   above its central prediction. Give one plausible reason for each miss.
5. Name the four modeling lanes from Submission 1 and the one-line idea of each.

## 1. Your beam's test result

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
TAU_I = 43.9e6                    # printed-interface shear strength (Pa) — starting
                                  # guess = bulk yield / sqrt(3)     (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def estimated_mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_est_g"] = estimated_mass_g(df.b, df.H)
df["mass_delta_g"] = df.weight_g - df.mass_est_g
df["mass_delta_pct"] = 100*df.mass_delta_g/df.mass_est_g
print(len(df), "tested beams")

new = pd.DataFrame([
    dict(beam_id=15, b=1.25, H=13.2, strength_N=515.6,
         failure_note="equation-query result; observed morphology not supplied"),
    dict(beam_id=16, b=1.44, H=13.2, strength_N=533.1,
         failure_note="locked-GP-query result; observed morphology not supplied"),
])
new["weight_g"] = np.nan
new["mass_est_g"] = estimated_mass_g(new.b, new.H)
df = pd.concat([df, new], ignore_index=True)
df["str_to_weight"] = df.strength_N / df.mass_est_g

# >>> ENTER your group's final design and its measured result:
b_mine, H_mine = None, None        # your Submission 1 design (mm)
P_mine = None                      # measured failure load (N)
mass_measured_mine = None          # measured printed-beam mass (g)
note_mine = ""                     # what the failure looked like
pred_median_sw = None              # copy model central prediction from Submission 1
pred_sigma_log = None              # copy epistemic sigma_log from Submission 1
if None not in (b_mine, H_mine, P_mine, mass_measured_mine,
                pred_median_sw, pred_sigma_log):
    mass_est_mine = estimated_mass_g(b_mine, H_mine)
    sw_mine_model_basis = P_mine / mass_est_mine
    sw_mine_measured_mass = P_mine / mass_measured_mine
    sigma_total_log = np.sqrt(pred_sigma_log**2 + 0.03**2)
    pred_lo_sw = pred_median_sw*np.exp(-2*sigma_total_log)
    pred_hi_sw = pred_median_sw*np.exp(2*sigma_total_log)
    inside_2sigma = pred_lo_sw <= sw_mine_model_basis <= pred_hi_sw
    print(f"your beam: ({b_mine}, {H_mine}), {P_mine} N")
    print("  observed failure note:", note_mine)
    print(f"  measured mass={mass_measured_mine:.2f} g; "
          f"estimated mass={mass_est_mine:.2f} g; "
          f"difference={mass_measured_mine-mass_est_mine:+.2f} g")
    print(f"  measured-mass str/w={sw_mine_measured_mass:.1f} N/g; "
          f"model-basis str/w={sw_mine_model_basis:.1f} N/g")
    print(f"  sigma_epi={pred_sigma_log:.3f}, sigma_total={sigma_total_log:.3f}")
    print(f"posterior-predictive interval: [{pred_lo_sw:.1f}, {pred_hi_sw:.1f}] N/g")
    print("inside recorded model +/-2 sigma interval:", inside_2sigma)
    print(f"class scoreboard: best tested so far {df.str_to_weight.max():.1f} N/g")

The model was trained on strength divided by estimated mass, so the interval
comparison uses that same denominator. Report the measured-mass ratio too, but
do not compare ratios with different denominators as if they were the same
quantity. If the result lies outside the interval, distinguish model-form error,
print-to-print variability, and an unmodeled failure mechanism. One test does
not identify which.

## 2. The lightweight challenge: hold 700 N, weigh as little as possible

Same 16 beams, same tools — different objective. Now strength is a
**constraint**, not the prize. The class-default confidence rule: require the
model's lower posterior-predictive quantile for one future beam to clear the
target. Epistemic uncertainty and 3% aleatory observation scatter are
independent in log space:

$$\sigma_{total}=\sqrt{\sigma_{epi}^2+0.03^2},\qquad
P_{lo}(b,H)=e^{\mu_{\ln P}(b,H)-2\sigma_{total}(b,H)}
\ge 700\text{ N}.$$

Among designs that pass, take the lightest. **FILL IN** the two marked lines.
(You may argue a different z than 2 in your memo — that is a risk posture,
not a math fact.)

In [ ]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    def J_rect(x, y):
        short, long = min(x, y), max(x, y)
        r = short/long
        beta = 1 - 0.63*r + 0.052*r**5
        return (1/3)*beta*long*short**3
    J = J_rect(b_, h_) + 2*J_rect(tf_, B_)
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c,
                b=b_, h=h_, tf=tf_, B=B_)

def P_bend(p, sy):
    return 4*sy*p["Ix"] / (p["c"] * L/1e3)
def P_shear(p, sy, cs):
    return 2 * (cs*sy/np.sqrt(3)) * p["b"]*p["h"]
def P_interaction_surrogate(Pb, Ps):
    return 1.0/np.sqrt(1/Pb**2 + 1/Ps**2)
def P_pointwise_yield(p, sy, n=801, return_detail=False):
    """Elastic first yield from co-located My/I and VQ/(It) stresses."""
    c, h2 = p["c"], p["h"]/2
    eps = max(c, 1.0)*1e-10
    y = np.unique(np.r_[np.linspace(0, c, n),
                        max(0, h2-eps), min(c, h2+eps)])
    in_web = y <= h2
    width = np.where(in_web, p["b"], p["B"])
    q_flange = p["B"]*p["tf"]*(h2 + p["tf"]/2)
    Q = np.where(
        in_web,
        q_flange + p["b"]*(h2-y)*(y+h2)/2,
        p["B"]*(c-y)*(y+c)/2,
    )
    sigma_per_N = (L/1e3)*y/(4*p["Ix"])
    tau_per_N = Q/(2*p["Ix"]*width)
    vm_per_N = np.sqrt(sigma_per_N**2 + 3*tau_per_N**2)
    loads = np.divide(sy, vm_per_N, out=np.full_like(vm_per_N, np.inf),
                      where=vm_per_N > 0)
    i = int(np.argmin(loads))
    if return_detail:
        return float(loads[i]), float(y[i]), float(sigma_per_N[i]), float(tau_per_N[i])
    return float(loads[i])
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)
def Q_flange(p):
    return p["B"]*p["tf"]*(p["h"]/2 + p["tf"]/2)
def P_sep(p, tau_i):
    """Flange-web separation: junction shear flow vs printed-interface strength."""
    return 2*tau_i*p["Ix"]*p["b"]/Q_flange(p)
def capacity(b, H, sy, k, tau_i):
    """Class model (2026-07-15): plain minimum of the three mode capacities."""
    p = section_props(b, H)
    return min(P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k))
def gov_mode(b, H, sy, k, tau_i):
    """Dominant pure-mode proxy, not an observed failure-mechanism label."""
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k)
    return "separation" if Ps < min(Pb, Pl) else (
        "LTB" if Pl < 0.999*Pb else "bend")

SY_CAL, K_CAL, TAU_CAL = 6.680e+07, 0.377, 1.791e+07
P_TARGET = 700.0

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF

# class-default model: lane A (log str/w), 3% noise — swap in your own lane if you prefer
X = df[["b", "H"]].values
fmu, fsd = X.mean(0), X.std(0) + 1e-12
y = np.log(df.str_to_weight.values)
ymean = y.mean()
gp = GaussianProcessRegressor(C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
                              alpha=0.03**2, normalize_y=False,
                              n_restarts_optimizer=5, random_state=0).fit((X-fmu)/fsd, y-ymean)

bg = np.linspace(1.25, 7.0, 60); Hg = np.linspace(5.0, 16.0, 60)
BB, HH = np.meshgrid(bg, Hg)
Xg = np.column_stack([BB.ravel(), HH.ravel()])
mu_c, std = gp.predict((Xg - fmu)/fsd, return_std=True)
mass_grid = estimated_mass_g(Xg[:, 0], Xg[:, 1])
# strength = str/w * mass, so in logs: ln P = (mu + ymean) + ln(mass)
mu_lnP = mu_c + ymean + np.log(mass_grid)
sigma_aleatory = 0.03
sigma_total = np.sqrt(std**2 + sigma_aleatory**2)

P_lo = ____        # >>> FILL IN: lower predictive strength, exp(mu_lnP minus 2*sigma_total)
feasible = ____    # >>> FILL IN: boolean mask, P_lo at or above P_TARGET
median_strength = np.exp(mu_lnP)
masked = np.where(feasible, mass_grid, np.inf)
i = int(np.argmin(masked))
b_lt, H_lt = float(Xg[i, 0]), float(Xg[i, 1])
median_feasible = median_strength >= P_TARGET
i_median = int(np.argmin(np.where(median_feasible, mass_grid, np.inf)))
lighter_infeasible = (~feasible) & (mass_grid < mass_grid[i])
if lighter_infeasible.any():
    j = int(np.argmax(np.where(lighter_infeasible, mass_grid, -np.inf)))
else:
    j = None
print(f"LIGHTWEIGHT DESIGN: b = {b_lt:.2f} mm, H_web = {H_lt:.2f} mm")
print(f"  mass {mass_grid[i]:.1f} g,  P_lo {P_lo[i]:.0f} N,  "
      f"posterior median {median_strength[i]:.0f} N")
print(f"  uncertainty allowance: posterior median - P_lo = "
      f"{median_strength[i]-P_lo[i]:.0f} N")
print(f"  median-only lightest design: b={Xg[i_median,0]:.2f}, "
      f"H_web={Xg[i_median,1]:.2f}, mass={mass_grid[i_median]:.1f} g, "
      f"median={median_strength[i_median]:.0f} N, P_lo={P_lo[i_median]:.0f} N")
print(f"  mass added by the 2-sigma rule versus median-only: "
      f"{mass_grid[i]-mass_grid[i_median]:.1f} g")
if j is not None:
    print(f"  closest-in-mass lighter infeasible grid point: b={Xg[j,0]:.2f}, "
          f"H_web={Xg[j,1]:.2f}, mass={mass_grid[j]:.3f} g "
          f"({mass_grid[i]-mass_grid[j]:.3f} g lighter), P_lo={P_lo[j]:.0f} N")
print(f"  calibrated-physics check: {capacity(b_lt, H_lt, SY_CAL, K_CAL, TAU_CAL):.0f} N, "
      f"mode {gov_mode(b_lt, H_lt, SY_CAL, K_CAL, TAU_CAL)}")
print("\nCHECKPOINT (class-default model): you should arrive at "
      "b = 4.56, H_web = 13.20, mass = 22.1 g.")
print("If you are not getting that, check your work or talk to a TA.")

### Stress-test the assumed aleatory noise

The table below refits the same class-default GP at 1%, 3%, and 10% observation
noise and uses that same value in each future-beam predictive bound. This is a
sensitivity analysis, not a vote on which noise value is true.

In [ ]:
noise_design_rows = []
for pct in [1, 3, 10]:
    r = pct/100
    gp_r = GaussianProcessRegressor(
        C(1.0, (1e-3, 1e3))*RBF([1.0, 1.0], (1e-1, 30.0)),
        alpha=r**2, normalize_y=False,
        n_restarts_optimizer=5, random_state=0).fit((X-fmu)/fsd, y-ymean)
    mu_r, epi_r = gp_r.predict((Xg-fmu)/fsd, return_std=True)
    total_r = np.sqrt(epi_r**2 + r**2)
    lo_r = np.exp(mu_r+ymean+np.log(mass_grid)-2*total_r)
    i_r = int(np.argmin(np.where(lo_r >= P_TARGET, mass_grid, np.inf)))
    noise_design_rows.append(dict(
        noise_pct=pct, b=Xg[i_r, 0], H_web=Xg[i_r, 1],
        mass_est_g=mass_grid[i_r],
        median_strength_N=np.exp(mu_r[i_r]+ymean)*mass_grid[i_r],
        lower_predictive_N=lo_r[i_r]))
noise_design_table = pd.DataFrame(noise_design_rows)
print(noise_design_table.round(2).to_string(index=False))

## Memo

1. Reflection: report measured and estimated mass, use the model-basis ratio for
   the interval check, and compare the observed failure note with the modeled proxy.
2. Margin: use the printed posterior median, lower bound, median-only design,
   robust design, and closest-in-mass lighter infeasible candidate. State the
   comparison in newtons and grams.
3. `z`: defend 2 or price another value. Under the independent Gaussian
   log-noise model, the chance that one future measured beam falls below a
   two-sigma lower predictive bound is 2.28%. Kernel and model-form errors are
   outside that probability statement.
4. Physics veto: cite the calibrated capacity and dominant-mode proxy. If it
   disagrees with the GP constraint, explain which evidence you prioritize.
5. Noise sensitivity: use the 1/3/10% table. State what moves and whether your
   design decision is assumption-sensitive.
6. One more test: provide coordinates and say whether posterior median,
   epistemic sigma, or proximity to the feasibility boundary motivates it.